# Reconcile all reactor cases

This notebook is intended to live in `examples/`. It finds all reactor YAML files under `../reactors/`, runs each case in `reconcile` mode, and renders a variable-by-reactor comparison table.

Cell colors:

- **Green**: verified and unchanged, or computed with no initial reference.
- **Yellow**: verified and changed only within the variable tolerance. The cell shows `initial (reconciled)`.
- **Red**: not verified, or changed beyond tolerance. The cell shows `initial -> reconciled`.
- **Gray**: unused, undecided, or unavailable.

In [1]:
from __future__ import annotations

from pathlib import Path
import sys
import traceback
import importlib
import pkgutil
from typing import Any

import numpy as np
import pandas as pd
from IPython.display import display, HTML

# The notebook is expected to be in examples/.
NOTEBOOK_DIR = Path.cwd().resolve()
PROJECT_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "examples" else NOTEBOOK_DIR
SRC_DIR = PROJECT_ROOT / "src"
REACTORS_DIR = PROJECT_ROOT / "reactors"

for path in (str(SRC_DIR), str(PROJECT_ROOT)):
    if path not in sys.path:
        sys.path.insert(0, path)

print(f"Project root: {PROJECT_ROOT}")
print(f"Reactors dir: {REACTORS_DIR}")

Project root: /home/alessmor/Scrivania/fusdb
Reactors dir: /home/alessmor/Scrivania/fusdb/reactors


In [ ]:
# Import FusDB and all relation modules so decorators populate the registries.
# If your package already imports all relations during normal startup, this is harmless.

from pathlib import Path
import sys
import traceback
import importlib
import pkgutil


def find_project_root(start: Path | None = None) -> Path:
    """Locate the repository root by searching for pyproject.toml."""
    current = (start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if (candidate / "pyproject.toml").exists():
            return candidate
    return current


PROJECT_ROOT = find_project_root()
SRC_DIR = PROJECT_ROOT / "src"
REACTORS_DIR = PROJECT_ROOT / "reactors"

for path in (str(SRC_DIR), str(PROJECT_ROOT)):
    if path not in sys.path:
        sys.path.insert(0, path)

from fusdb import Reactor
import fusdb


# Prefer packages whose names indicate they contain relations.
def import_relation_modules() -> list[tuple[str, str]]:
    """Import relation modules and return non-fatal import errors."""
    errors: list[tuple[str, str]] = []
    visited: set[str] = set()

    root_names = []
    for candidate in ("fusdb.relations", "fusdb.relation", "fusdb.models"):
        try:
            importlib.import_module(candidate)
            root_names.append(candidate)
            visited.add(candidate)
        except Exception:
            pass

    # Walk selected relation packages first.
    for root_name in root_names:
        module = sys.modules.get(root_name)
        module_path = getattr(module, "__path__", None)
        if module_path is None:
            continue
        for info in pkgutil.walk_packages(module_path, root_name + "."):
            name = info.name
            if name in visited:
                continue
            visited.add(name)
            try:
                importlib.import_module(name)
            except Exception as exc:
                errors.append((name, repr(exc)))

    # Fallback: walk fusdb and import modules whose names likely register relations.
    for info in pkgutil.walk_packages(fusdb.__path__, fusdb.__name__ + "."):
        name = info.name
        if name in visited or ".tests" in name:
            continue
        if not any(part in name.lower() for part in ("relation", "physics", "geometry", "power", "plasma", "confinement")):
            continue
        visited.add(name)
        try:
            importlib.import_module(name)
        except Exception as exc:
            errors.append((name, repr(exc)))

    return errors


print(f"Project root: {PROJECT_ROOT}")
print(f"Reactors dir: {REACTORS_DIR}")

import_errors = import_relation_modules()
if import_errors:
    print("Non-fatal relation import errors:")
    for name, err in import_errors[:20]:
        print(f"  {name}: {err}")
    if len(import_errors) > 20:
        print(f"  ... {len(import_errors) - 20} more")
else:
    print("Relation modules imported without reported errors.")

ModuleNotFoundError: No module named 'fusdb'

In [ ]:
def find_reactor_files(reactors_dir: Path) -> list[Path]:
    """Find reactor YAML files below reactors/."""
    if not reactors_dir.exists():
        raise FileNotFoundError(f"Reactors directory not found: {reactors_dir}")

    files = sorted(reactors_dir.rglob("reactor.yaml"))
    files += sorted(reactors_dir.rglob("reactor.yml"))

    # Fallback for flat layouts where each reactor is one YAML file.
    if not files:
        files = sorted(reactors_dir.rglob("*.yaml")) + sorted(reactors_dir.rglob("*.yml"))

    # Avoid duplicate paths if reactor.yaml also matched fallback in edited notebooks.
    unique = []
    seen = set()
    for path in files:
        resolved = path.resolve()
        if resolved not in seen:
            seen.add(resolved)
            unique.append(path)
    return unique


reactor_files = find_reactor_files(REACTORS_DIR)
print(f"Found {len(reactor_files)} reactor file(s):")
for path in reactor_files:
    print(" -", path.relative_to(PROJECT_ROOT))

In [ ]:
def run_one_reactor(path: Path) -> dict[str, Any]:
    """Load one reactor, build its relation system, and run reconcile mode."""
    record: dict[str, Any] = {
        "path": path,
        "name": path.parent.name,
        "reactor": None,
        "system": None,
        "result": None,
        "error": None,
        "traceback": None,
    }
    try:
        reactor = Reactor.from_yaml(path)
        system = reactor.to_relation_system()
        result = system.run("reconcile")
        record.update({
            "name": reactor.name,
            "reactor": reactor,
            "system": system,
            "result": result,
        })
    except Exception as exc:
        record["error"] = repr(exc)
        record["traceback"] = traceback.format_exc()
    return record


records = [run_one_reactor(path) for path in reactor_files]

summary_rows = []
for rec in records:
    result = rec["result"] or {}
    system = rec["system"]
    summary_rows.append({
        "reactor": rec["name"],
        "path": str(rec["path"].relative_to(PROJECT_ROOT)),
        "success": result.get("success", False) if result else False,
        "termination": result.get("termination", rec["error"] or "not run"),
        "errors": len(result.get("errors", [])) if result else 1,
        "warnings": len(result.get("warnings", [])) if result else 0,
        "active_relations": len(getattr(system, "relations", result.get("relations", []))) if system is not None else 0,
        "blocked_relations": len(getattr(system, "blocked_relations", [])) if system is not None else 0,
    })

summary = pd.DataFrame(summary_rows).set_index("reactor") if summary_rows else pd.DataFrame()
display(summary)

failed = [rec for rec in records if rec["error"]]
if failed:
    print("\nLoad/run failures:")
    for rec in failed:
        print(f"\n{rec['name']} ({rec['path']}): {rec['error']}")
        print(rec["traceback"])

In [ ]:
def to_array(value: Any) -> np.ndarray | None:
    """Convert a scalar/profile-like value to a float array, or None if unavailable."""
    if value is None:
        return None
    try:
        arr = np.asarray(value, dtype=float)
    except Exception:
        return None
    if arr.size == 0 or not np.all(np.isfinite(arr)):
        return None
    return arr


def max_relative_error(reference: Any, value: Any) -> float | None:
    """Return max pointwise relative error between reference and value."""
    ref = to_array(reference)
    val = to_array(value)
    if ref is None or val is None:
        return None
    try:
        ref, val = np.broadcast_arrays(ref, val)
    except ValueError:
        return None
    denom = np.maximum(np.maximum(np.abs(ref), np.abs(val)), 1.0)
    return float(np.max(np.abs(val - ref) / denom))


def is_exact(reference: Any, value: Any) -> bool:
    """Exact numerical equality for scalars/profiles."""
    ref = to_array(reference)
    val = to_array(value)
    if ref is None or val is None:
        return False
    try:
        ref, val = np.broadcast_arrays(ref, val)
    except ValueError:
        return False
    return bool(np.array_equal(ref, val))


def fmt_number(x: float) -> str:
    """Compact numeric formatting."""
    if x == 0:
        return "0"
    ax = abs(x)
    if ax >= 1e4 or ax < 1e-3:
        return f"{x:.4e}"
    return f"{x:.6g}"


def summarize_value(value: Any, *, max_items: int = 3) -> str:
    """Compact display for scalar or profile values."""
    if value is None:
        return ""
    arr = to_array(value)
    if arr is None:
        return str(value)
    if arr.ndim == 0 or arr.size == 1:
        return fmt_number(float(arr.ravel()[0]))
    flat = arr.ravel()
    if flat.size <= max_items:
        items = ", ".join(fmt_number(float(x)) for x in flat)
        return f"[{items}]"
    return (
        f"array{tuple(arr.shape)} "
        f"mean={fmt_number(float(np.mean(flat)))}, "
        f"min={fmt_number(float(np.min(flat)))}, "
        f"max={fmt_number(float(np.max(flat)))}"
    )


def classify_variable(var: Any, status: str | None) -> tuple[str, str]:
    """Return (display_text, color_class) for one variable."""
    value = getattr(var, "value", None)
    reference = getattr(var, "reference_value", None)
    rel_tol = float(getattr(var, "rel_tol", 1e-2) or 1e-2)
    status = status or getattr(var, "validity", None) or "unknown"

    value_text = summarize_value(value)
    reference_text = summarize_value(reference)

    if value is None and reference is None:
        return "", "blank"

    if status in {"unused", "undecided"}:
        return value_text or reference_text, "unused"

    if reference is None:
        if status in {"consistent", "verified", "adjusted"}:
            return f"computed: {value_text}", "exact"
        return value_text, "wrong"

    err = max_relative_error(reference, value)
    exact = is_exact(reference, value)

    if status in {"suspect", "fixed_inconsistent", "unresolved", "invalid", "missing"}:
        return f"{reference_text} -> {value_text}", "wrong"

    if exact:
        return value_text, "exact"

    if err is not None and err <= rel_tol:
        return f"{reference_text} ({value_text})", "within"

    return f"{reference_text} -> {value_text}", "wrong"

In [ ]:
# Build variable-by-reactor comparison table.

successful_records = [rec for rec in records if rec["result"] is not None]
reactor_names = [rec["name"] for rec in successful_records]

all_variable_names: list[str] = []
seen_variables: set[str] = set()
for rec in successful_records:
    variables = (rec["result"] or {}).get("variables", {})
    for name in variables:
        if name not in seen_variables:
            seen_variables.add(name)
            all_variable_names.append(name)
all_variable_names = sorted(all_variable_names)

values_df = pd.DataFrame(index=all_variable_names, columns=reactor_names, dtype=object)
classes_df = pd.DataFrame(index=all_variable_names, columns=reactor_names, dtype=object)

for rec in successful_records:
    name = rec["name"]
    result = rec["result"] or {}
    variables = result.get("variables", {})
    variable_status = result.get("variable_status", {})
    for var_name in all_variable_names:
        var = variables.get(var_name)
        if var is None:
            values_df.loc[var_name, name] = ""
            classes_df.loc[var_name, name] = "blank"
            continue
        text, cls = classify_variable(var, variable_status.get(var_name))
        values_df.loc[var_name, name] = text
        classes_df.loc[var_name, name] = cls


def style_cells(_data: pd.DataFrame) -> pd.DataFrame:
    styles = pd.DataFrame("", index=values_df.index, columns=values_df.columns)
    styles[classes_df == "exact"] = "background-color: #c6efce; color: #006100"
    styles[classes_df == "within"] = "background-color: #ffeb9c; color: #9c6500"
    styles[classes_df == "wrong"] = "background-color: #ffc7ce; color: #9c0006"
    styles[classes_df == "unused"] = "background-color: #eeeeee; color: #666666"
    styles[classes_df == "blank"] = "background-color: #f8f8f8; color: #999999"
    return styles


styled_table = (
    values_df
    .style
    .apply(style_cells, axis=None)
    .set_properties(**{"white-space": "pre-wrap", "font-size": "0.85em"})
)

display(HTML("<h3>Variable reconciliation table</h3>"))
display(styled_table)

In [ ]:
# Optional: save the styled table as standalone HTML beside this notebook.

output_html = NOTEBOOK_DIR / "reconcile_all_reactors_results.html"
with output_html.open("w", encoding="utf-8") as handle:
    handle.write(styled_table.to_html())

print(f"Saved styled table to: {output_html}")